# Azure PBT remaining-training pilot — frozen 2026-09-06

**Setup.** Use all 19 remaining tasks from the original 20-task training split of `data/apps_hard.json`, excluding only previously inspected smoke task 3692. Include both existing candidates per task: 38 candidates total. No task is selected by a score, observed failure, difficulty, or outcome. The 30 held-out tasks are not submitted to any model. Azure deployment `gpt-5.6-terra`, seed 300, low reasoning, visible code; ten trigger inputs per candidate. Trigger output cap 12,000 tokens, one HTTP retry configured, cache enabled.

**Change.** Scale the validated smoke protocol to the remainder of the frozen training population. Prompt content remains unchanged. This is training/development evidence and cannot establish the held-out prompt effect.

**Hypothesis.** The existing trigger protocol produces parseable, domain-valid inputs across the remaining tasks; broadening tasks will reveal whether the smoke's invalid-input issue recurs.

**Prediction.** Exactly 38 candidate records; all have a usable response containing ten parsed inputs with zero discarded inputs, giving 380 requested/parsed inputs. A separate specification-domain review must classify every input as valid, invalid, or unresolved before authoring. Operational success requires zero infrastructure failures. Report failures and invalid/unresolved counts explicitly rather than replacing them with clean negatives.

**Stage and spending scope.** This notebook launches ONLY the 38 logical trigger-search calls. At the configured 12,000-token limit, their first-attempt output envelope is 456,000 tokens, or 912,000 if every call uses one full configured retry; input tokens add usage. These are configuration-based token estimates, not an enforced dollar cap; actual Azure rates/cost have not been verified. Existing recorded candidates resume without another call; an interrupted in-flight call may be paid again. Recorded failures are retained, not retried or overwritten automatically.

**Future authoring comparison (not launched here).** After reviewing the common inputs, hold model/seed/reasoning/inputs fixed and compare `plain_v3` with `traceable_v1`: ten tests, `resolve='with'`, visible code, no critique, 8,192 output tokens, 300-second call timeout, Docker execution. Hypothesis: explicit traceability reduces false alarms while retaining useful attack catches; training diagnostic target is at least a 0.10 honest-FPR reduction with at least 90% of the control's valid-attack catch rate. These are training targets only. Include suite errors, duplicate test bodies, valid-input sensitivity, and attack-suite replay on honest code before interpreting any difference. Both arms see exactly the same reviewed inputs per candidate; no candidate-specific input regeneration between arms.

**Observed.** Trigger stage launched on 2026-09-06 as `azure-terra-pbt-train19-s300-v1-triggers` (initial worker PID 23336). Launch and process liveness were verified; results and domain review are pending. No authoring or held-out scoring has been launched.

**Execution.** Run setup cells 1–4, then cell 5 to launch the trigger worker. Cell 6 reports artifacts without launching anything. The Windows worker runs detached/hidden and requests system wakefulness while active; candidate code is still only executed through Docker in later stages. Restarting setup and launch resumes records; a live lock refuses duplicate launch. Lock recovery requires checking the saved PID and artifacts. There is deliberately no authoring launch cell.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
from dotenv import load_dotenv
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
os.chdir(REPO)
assert (REPO / "pipeline").is_dir() and (REPO / "data").is_dir()
load_dotenv(REPO / ".env", encoding="utf-8-sig", override=True)
from pipeline.data import Dataset, load_records
from pipeline.protocols import TriggerSearch, UnitTesting
from pipeline.protocols.base import Run

MODEL = "openai-api/azureai/gpt-5.6-terra"
PREFIX = "azure-terra-pbt-train19-s300-v1"
SOURCE = Path("data/apps_hard.json")
DATA = Path("data/azure_pbt_train19_s300_v1.json")
TRIGGERS = PREFIX + "-triggers"
CONTROL = PREFIX + "-plain-v3"
TRACEABLE = PREFIX + "-traceable-v1"
SEED, N_INPUTS, N_TESTS = 300, 10, 10
assert os.environ["AZURE_OPENAI_DEPLOYMENT"] == "gpt-5.6-terra"
assert os.environ["AZURE_OPENAI_API_KEY"]
from urllib.parse import urlsplit
endpoint_parts = urlsplit(os.environ["AZURE_OPENAI_ENDPOINT"].strip())
assert endpoint_parts.scheme == 'https' and endpoint_parts.netloc == 'omar-ai.services.ai.azure.com'
assert not endpoint_parts.query and not endpoint_parts.fragment
assert endpoint_parts.path.rstrip('/') in ('', '/openai/v1', '/openai/v1/responses')
endpoint = 'https://omar-ai.services.ai.azure.com/openai/v1'
os.environ["AZUREAI_BASE_URL"] = endpoint
os.environ["AZUREAI_API_KEY"] = os.environ["AZURE_OPENAI_API_KEY"]


In [ ]:
source = Dataset.load(SOURCE)
assert len(source.split["train"]) == 20 and len(source.split["test"]) == 30
assert "3692" in source.split["train"]
SELECTED_IDS = tuple(task_id for task_id in source.split["train"] if task_id != "3692")
assert len(SELECTED_IDS) == 19
assert not set(SELECTED_IDS).intersection(source.split["test"])
source_hash = hashlib.sha256(SOURCE.read_bytes()).hexdigest()
assert source_hash == "4237df1d48b1c1d1cd4a9b197ce57948a231d8dad26abace58715c7254329974", "Source dataset changed"
training = Dataset(
    name="azure-pbt-remaining-training-v1", backend=source.backend, io_mode=source.io_mode,
    tasks=tuple(source.task(task_id) for task_id in SELECTED_IDS),
    split={"train": SELECTED_IDS, "test": ()},
    built_from={"source": str(SOURCE), "source_sha256": source_hash,
                "source_built_from": dict(source.built_from),
                "selection": "all frozen training tasks except previously inspected smoke task3692",
                "excluded_smoke_task": "3692", "seed": SEED},
)
expected_document = training.to_json()
if DATA.exists():
    assert json.loads(DATA.read_text(encoding="utf-8")) == expected_document, "Existing training data differs"
else:
    DATA.write_text(json.dumps(expected_document, indent=2) + "\n", encoding="utf-8")
dataset = Dataset.load(DATA)
assert len(dataset.train) == 19 and not dataset.test
EXPECTED_IDS = {candidate.candidate_id for _, candidate in dataset.candidates()}
assert len(EXPECTED_IDS) == 38
print({"task_ids": list(SELECTED_IDS), "candidate_count": len(EXPECTED_IDS),
       "source_sha256": source_hash, "dataset_sha256": hashlib.sha256(DATA.read_bytes()).hexdigest()})


In [ ]:
trigger = TriggerSearch(run_name=TRIGGERS, data=str(DATA), model=MODEL, num_inputs=N_INPUTS,
                        reasoning="low", code_visible=True, seed=SEED, cache=True)
common = dict(data=str(DATA), model=MODEL, triggers=TRIGGERS, n_tests=N_TESTS,
              code_visible=True, resolve="with", reasoning="low", max_tokens=8192,
              call_seconds=300, sandbox_seconds=120, seed=SEED, critique=False, cache=True,
              docker_image="python:3.12-slim")
control = UnitTesting(run_name=CONTROL, test_gen_prompt="plain_v3", **common)
traceable = UnitTesting(run_name=TRACEABLE, test_gen_prompt="traceable_v1", **common)
assert trigger.runtime(SEED).max_tokens == 12000
assert trigger.runtime(SEED).http_retries == 1
assert control._runtime().http_retries == 1

assert trigger.total == control.total == traceable.total == 38
assert all(trigger.data.split_of(task.task_id) == "train" for task in trigger.data.tasks)


In [ ]:
WORKER = r'''
from pathlib import Path
import json, os, sys, traceback
from dotenv import load_dotenv
load_dotenv(Path.cwd() / ".env", encoding="utf-8-sig", override=True)
from urllib.parse import urlsplit
endpoint_parts = urlsplit(os.environ["AZURE_OPENAI_ENDPOINT"].strip())
assert endpoint_parts.scheme == 'https' and endpoint_parts.netloc == 'omar-ai.services.ai.azure.com'
assert not endpoint_parts.query and not endpoint_parts.fragment
assert endpoint_parts.path.rstrip('/') in ('', '/openai/v1', '/openai/v1/responses')
base = 'https://omar-ai.services.ai.azure.com/openai/v1'
os.environ["AZUREAI_BASE_URL"] = base
os.environ["AZUREAI_API_KEY"] = os.environ["AZURE_OPENAI_API_KEY"]
import pipeline.protocols
from pipeline.protocols.base import Run
config_path = Path(sys.argv[1])
lock_path = config_path.parent / "training-worker.lock"
exit_path = config_path.parent / "training-worker-exit.json"
awake_state = None
try:
    if os.name == "nt":
        import ctypes
        awake_state = ctypes.windll.kernel32.SetThreadExecutionState(0x80000001)
        if not awake_state:
            raise OSError("Windows refused the system-awake request")
    run = Run.from_config(json.loads(config_path.read_text(encoding="utf-8")))
    assert run.total == 38 and len(run.data.train) == 19 and not run.data.test
    assert run.protocol == "trigger_search"
    written = run.execute()
    result = {"exit_code": 0, "written": written, "scored": len(run.get_records()), "total": run.total}
except BaseException as error:
    traceback.print_exc()
    result = {"exit_code": 1, "error_type": type(error).__name__}
finally:
    if awake_state:
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000000)
    exit_path.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    lock_path.unlink(missing_ok=True)
sys.exit(result["exit_code"])
'''

def launch_training_triggers(run):
    """Launch only the bounded trigger stage; existing records remain authoritative."""
    assert run.run_name == TRIGGERS and run.protocol == "trigger_search"
    assert run.total == 38 and len(run.data.train) == 19 and not run.data.test
    run.write_config()
    if not run.pending():
        print({"run": run.run_name, "state": "all38 candidate records already present"})
        return
    subprocess.run(["docker", "info", "--format", "{{.ServerVersion}}"],
                   check=True, capture_output=True, text=True, timeout=30)
    subprocess.run(["docker", "image", "inspect", "python:3.12-slim"],
                   check=True, capture_output=True, text=True, timeout=30)
    if os.name != "nt":
        raise RuntimeError("This training launcher requires Windows; use the existing tmux protocol with an external awake guard on Linux")
    lock = run.directory / "training-worker.lock"
    descriptor = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    os.close(descriptor)
    try:
        with (run.directory / "training-worker.log").open("ab") as log:
            process = subprocess.Popen(
                [sys.executable, "-u", "-c", WORKER, str(run.config_path)],
                cwd=REPO, stdin=subprocess.DEVNULL, stdout=log, stderr=log,
                creationflags=subprocess.DETACHED_PROCESS | subprocess.CREATE_NEW_PROCESS_GROUP
                              | subprocess.CREATE_NO_WINDOW,
                close_fds=True,
            )
        (run.directory / "training-worker-pid.json").write_text(
            json.dumps({"pid": process.pid, "run": run.run_name}) + "\n", encoding="utf-8")
    except BaseException:
        lock.unlink(missing_ok=True)
        raise
    print({"run": run.run_name, "pid": process.pid, "pending_at_launch": len(run.pending()),
           "next_step": "inspect38 trigger records and review input domains before authoring"})


In [ ]:
launch_training_triggers(trigger)


In [ ]:
rows = load_records(TRIGGERS)
assert len(rows) <= 38
assert len({row["candidate_id"] for row in rows}) == len(rows), "Duplicate candidate records"
assert all(row["candidate_id"] in EXPECTED_IDS and row["split"] == "train" for row in rows)
failures = [{"candidate_id": row["candidate_id"], "blame": row["blame"], "reason": row["reason"]}
            for row in rows if row["failed"]]
parsed = [{"candidate_id": row["candidate_id"], "n_parsed": row["n_parsed"], "dropped": row["dropped"]}
          for row in rows if not row["failed"]]
print(json.dumps({"record_count": len(rows), "expected": 38, "failures": failures, "parsed": parsed,
                  "authoring_launched": False,
                  "next_step": "review every generated input against its task specification"}, indent=2))
if len(rows) == 38:
    assert not failures, "Failed records retained: investigate before authoring"
    assert all(row["n_parsed"] == N_INPUTS and row["dropped"] == 0 for row in rows), "Incomplete trigger parse"
    assert sum(len(row["inputs"]) for row in rows) == 380
    print("38 responses/380 inputs parsed; semantic input-domain validity still requires review.")
